# 05 - Train Clause Classifier with LegalBERT

## Stage 6 Objective

Fine-tune `nlpaueb/legal-bert-base-uncased` on `data/processed/classification_dataset.csv` for `clause_type` classification.

If LegalBERT cannot be downloaded or loaded, the notebook falls back to `distilbert-base-uncased`. Use the fallback only when the primary model is unavailable or when hardware constraints require it.

## Outputs

- Model and tokenizer: `models/clause_classifier/`
- Label mapping: `models/clause_classifier/label_mapping.json`
- Predictions: `outputs/predictions/clause_classifier_predictions.csv`
- Confusion matrix CSV: `outputs/charts/clause_classifier_confusion_matrix.csv`
- Confusion matrix chart: `outputs/charts/clause_classifier_confusion_matrix.png`

## 1. Configure Paths and Training Settings

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classifier import (
    FALLBACK_MODEL_NAME,
    PRIMARY_MODEL_NAME,
    build_confusion_matrix_frame,
    build_label_mappings,
    compute_classification_metrics,
    encode_labels,
    ensure_directory,
    load_tokenizer_and_model_with_fallback,
    normalize_split,
    save_label_mapping,
    softmax_confidence,
)

DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classification_dataset.csv'
MODEL_DIR = PROJECT_ROOT / 'models' / 'clause_classifier'
LABEL_MAPPING_PATH = MODEL_DIR / 'label_mapping.json'
TRAINING_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'clause_classifier_trainer'
PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'predictions' / 'clause_classifier_predictions.csv'
CONFUSION_MATRIX_CSV_PATH = PROJECT_ROOT / 'outputs' / 'charts' / 'clause_classifier_confusion_matrix.csv'
CONFUSION_MATRIX_PNG_PATH = PROJECT_ROOT / 'outputs' / 'charts' / 'clause_classifier_confusion_matrix.png'

FORCE_DISTILBERT_FALLBACK = False
MAX_LENGTH = 256
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
SEED = 42

ensure_directory(MODEL_DIR)
ensure_directory(TRAINING_OUTPUT_DIR)
ensure_directory(PREDICTIONS_PATH.parent)
ensure_directory(CONFUSION_MATRIX_CSV_PATH.parent)

print(f'Dataset path: {DATASET_PATH}')
print(f'Model output directory: {MODEL_DIR}')
print(f'Primary model: {PRIMARY_MODEL_NAME}')
print(f'Fallback model: {FALLBACK_MODEL_NAME}')

## 2. Load and Validate Classification Dataset

In [ ]:
required_columns = ['clause_id', 'clause_text', 'clause_type', 'split']

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATASET_PATH}')

raw_df = pd.read_csv(DATASET_PATH)
missing_columns = [column for column in required_columns if column not in raw_df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

df = raw_df.copy()
df['clause_text'] = df['clause_text'].fillna('').astype(str).str.strip()
df['clause_type'] = df['clause_type'].fillna('').astype(str).str.strip()
df['split'] = df['split'].map(normalize_split)
df = df[(df['clause_text'] != '') & (df['clause_type'] != '')].reset_index(drop=True)

if df.empty:
    raise ValueError('No usable classification rows found.')
if not {'train', 'validation', 'test'}.issubset(set(df['split'])):
    raise ValueError('Dataset must contain train, validation, and test splits for Stage 6.')

print(f'Using {len(df)} classification row(s).')
display(df[['clause_id', 'split', 'clause_type', 'clause_text']].head(20))
display(df.groupby(['split', 'clause_type']).size().reset_index(name='count'))

## 3. Build Label Mapping

In [ ]:
label2id, id2label = build_label_mappings(df['clause_type'])
df['labels'] = encode_labels(df['clause_type'], label2id)

train_labels = set(df[df['split'] == 'train']['clause_type'])
all_labels = set(df['clause_type'])
missing_from_train = sorted(all_labels - train_labels)
if missing_from_train:
    print(f'Warning: labels not present in train split: {missing_from_train}. Add more labeled data before serious training.')

save_label_mapping(LABEL_MAPPING_PATH, label2id, id2label)
print(f'Saved label mapping to {LABEL_MAPPING_PATH.relative_to(PROJECT_ROOT)}')
display(pd.DataFrame({'clause_type': list(label2id.keys()), 'label_id': list(label2id.values())}))

## 4. Build Hugging Face Datasets

In [ ]:
from datasets import Dataset, DatasetDict

dataset_dict = DatasetDict(
    {
        split_name: Dataset.from_pandas(
            df[df['split'] == split_name].reset_index(drop=True),
            preserve_index=False,
        )
        for split_name in ['train', 'validation', 'test']
    }
)

dataset_dict

## 5. Load LegalBERT with DistilBERT Fallback

In [ ]:
import torch

primary_model_name = FALLBACK_MODEL_NAME if FORCE_DISTILBERT_FALLBACK else PRIMARY_MODEL_NAME
model_name, tokenizer, model = load_tokenizer_and_model_with_fallback(
    primary_model_name=primary_model_name,
    fallback_model_name=FALLBACK_MODEL_NAME,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Loaded model: {model_name}')
print(f'Using device: {device}')

## 6. Tokenize Clauses

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch['clause_text'],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)
keep_columns = {'input_ids', 'attention_mask', 'labels'}
columns_to_remove = [column for column in tokenized_datasets['train'].column_names if column not in keep_columns]
tokenized_datasets = tokenized_datasets.remove_columns(columns_to_remove)

tokenized_datasets

## 7. Configure Hugging Face Trainer

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def make_training_args():
    common_args = dict(
        output_dir=str(TRAINING_OUTPUT_DIR),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=0.01,
        logging_steps=1,
        save_strategy='epoch',
        save_total_limit=1,
        report_to=[],
        seed=SEED,
        fp16=False,
    )
    try:
        return TrainingArguments(eval_strategy='epoch', **common_args)
    except TypeError:
        return TrainingArguments(evaluation_strategy='epoch', **common_args)

training_args = make_training_args()

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)

try:
    trainer = Trainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = Trainer(tokenizer=tokenizer, **trainer_kwargs)

trainer

## 8. Fine-Tune the Classifier

In [ ]:
train_result = trainer.train()
train_result

## 9. Evaluate Validation and Test Performance

In [ ]:
validation_metrics = trainer.evaluate(eval_dataset=tokenized_datasets['validation'], metric_key_prefix='validation')
test_metrics = trainer.evaluate(eval_dataset=tokenized_datasets['test'], metric_key_prefix='test')

display(pd.DataFrame([validation_metrics, test_metrics]))

## 10. Save Model, Tokenizer, and Label Mapping

In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
save_label_mapping(LABEL_MAPPING_PATH, label2id, id2label)

print(f'Saved classifier model and tokenizer to {MODEL_DIR.relative_to(PROJECT_ROOT)}')
print(f'Saved label mapping to {LABEL_MAPPING_PATH.relative_to(PROJECT_ROOT)}')

## 11. Build Predictions for All Splits

In [ ]:
import numpy as np

prediction_frames = []
for split_name in ['train', 'validation', 'test']:
    split_df = df[df['split'] == split_name].reset_index(drop=True)
    if split_df.empty:
        continue
    output = trainer.predict(tokenized_datasets[split_name])
    predicted_ids = np.argmax(output.predictions, axis=-1)
    predicted_labels = [id2label[int(label_id)] for label_id in predicted_ids]
    confidences = softmax_confidence(output.predictions)

    prediction_frames.append(
        pd.DataFrame(
            {
                'clause_id': split_df['clause_id'],
                'split': split_df['split'],
                'clause_text': split_df['clause_text'],
                'true_clause_type': split_df['clause_type'],
                'predicted_clause_type': predicted_labels,
                'confidence': confidences,
            }
        )
    )

predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

print(f'Saved {len(predictions_df)} prediction row(s) to {PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')
display(predictions_df.head(20))

## 12. Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt

label_names = [id2label[index] for index in sorted(id2label)]
test_predictions_df = predictions_df[predictions_df['split'] == 'test'].copy()
if test_predictions_df.empty:
    print('No test predictions available for confusion matrix.')
else:
    confusion_df = build_confusion_matrix_frame(
        test_predictions_df['true_clause_type'],
        test_predictions_df['predicted_clause_type'],
        label_names,
    )
    confusion_df.to_csv(CONFUSION_MATRIX_CSV_PATH)
    display(confusion_df)

    fig, ax = plt.subplots(figsize=(max(6, len(label_names) * 1.2), max(4, len(label_names) * 0.9)))
    image = ax.imshow(confusion_df.values, cmap='Blues')
    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(label_names, rotation=45, ha='right')
    ax.set_yticklabels(label_names)
    ax.set_xlabel('Predicted clause_type')
    ax.set_ylabel('True clause_type')
    ax.set_title('Clause Classifier Confusion Matrix')
    for row_index in range(confusion_df.shape[0]):
        for col_index in range(confusion_df.shape[1]):
            ax.text(col_index, row_index, confusion_df.iloc[row_index, col_index], ha='center', va='center')
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(CONFUSION_MATRIX_PNG_PATH, dpi=150)
    plt.show()
    print(f'Saved confusion matrix CSV to {CONFUSION_MATRIX_CSV_PATH.relative_to(PROJECT_ROOT)}')
    print(f'Saved confusion matrix chart to {CONFUSION_MATRIX_PNG_PATH.relative_to(PROJECT_ROOT)}')